# Fase 4 - Actividad 6: Optimización y Entrenamiento Lineal Oficial

El objetivo central de este paso es la ejecución local de **GridSearchCV (K=5)** sobre el Pipeline completo (Submuestreo -> TF-IDF -> SVM/Logística). 


In [5]:
import pandas as pd
import joblib
import warnings
import tracemalloc
import time
warnings.filterwarnings('ignore')

from sklearn.model_selection import GridSearchCV
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from imblearn.pipeline import Pipeline
from imblearn.under_sampling import RandomUnderSampler

## 1. Carga del Dataset Completo de Entrenamiento
<!--  -->

In [6]:
input_path = 'data/train_val_set.csv'
df = pd.read_csv(input_path, sep=';')
df['texto_lineal'] = df['texto_lineal'].fillna('')

X = df[['texto_lineal']]
y = df['label']

print(f"Dimensiones de X para Entrenamiento Oficial (GridSearch): {X.shape}")

Dimensiones de X para Entrenamiento Oficial (GridSearch): (45784, 1)


## 2. Declaración del Pipeline y Malla de Hiperparámetros (Grid)

Montamos la estructura del pipeline e instruimos al `GridSearchCV` para que pruebe ambas familias de clasificadores lineales (Logística y SVM) con diferentes configuraciones.

In [7]:
# Nodos base
undersampler = RandomUnderSampler(random_state=42)
preprocessor = ColumnTransformer(
    transformers=[('tfidf', TfidfVectorizer(), 'texto_lineal')] 
)

# Pipeline molde
pipeline = Pipeline(steps=[
    ('undersampler', undersampler),
    ('vectorizer', preprocessor),
    ('classifier', LogisticRegression()) # Este clasificador será sustituido dinámicamente por la malla
])

# Malla combinada (Logística vs SVM Lineal)
param_grid = [
    {
        'vectorizer__tfidf__max_features': [5000, 10000],
        'classifier': [LogisticRegression(max_iter=1000, random_state=42)],
        'classifier__C': [0.1, 1.0, 10.0]
    },
    {
        'vectorizer__tfidf__max_features': [5000, 10000],
        'classifier': [LinearSVC(max_iter=2000, random_state=42)],
        'classifier__C': [0.1, 1.0, 10.0]
    }
]

# Configuración de Validación Cruzada (5 pliegues)
grid_search = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    cv=5,
    scoring='f1_macro',
    n_jobs=-1,
    verbose=2
)

print("GridSearchCV configurado con éxito.")

GridSearchCV configurado con éxito.


## 3. Entrenamiento Oficial y Captura de RAM Local

In [8]:
print("Iniciando GridSearchCV Oficial... (Esto tomará varios minutos)\n")

tracemalloc.start()
start_time = time.time()

# EJECUCIÓN DEL ENTRENAMIENTO CRUZADO Y BÚSQUEDA EXHAUSTIVA
grid_search.fit(X, y)

# DETENCIÓN DE SENSORES Y OBTENCIÓN DE LECTURAS
current, peak = tracemalloc.get_traced_memory()
end_time = time.time()
tracemalloc.stop()

# Conversión de lectura a Megabytes (MB)
peak_mb = peak / 10**6
tiempo_total = end_time - start_time

print("\n--- CAPTURA DE RENDIMIENTO FÍSICO EN ENTORNO LOCAL ---")
print(f"RAM Pico (Peak Memory) Consumida: {peak_mb:.2f} MB")
print(f"Tiempo de ejecución de la malla:  {tiempo_total:.2f} Segundos")

Iniciando GridSearchCV Oficial... (Esto tomará varios minutos)

Fitting 5 folds for each of 12 candidates, totalling 60 fits

--- CAPTURA DE RENDIMIENTO FÍSICO EN ENTORNO LOCAL ---
RAM Pico (Peak Memory) Consumida: 45.99 MB
Tiempo de ejecución de la malla:  42.47 Segundos


## 4. Obtención del Mejor Modelo y Exportación

Finalmente, el GridSearchCV nos revelará cuál fue el conjunto de hiperparámetros que logró el mayor F1-Score sin sobreajuste. Al tener `refit=True` por defecto, el ganador ya está entrenado con la totalidad del corpus y está listo para ser guardado.

In [10]:
print("\n--- RESULTADOS DEL GRIDSEARCH K=5 ---")
print(f"Mejor F1-Score Macro obtenido: {grid_search.best_score_:.4f}")

campeon = grid_search.best_estimator_.named_steps['classifier']
print(f"\nModelo Ganador: {type(campeon).__name__}")
print("Hiperparámetros óptimos:")
for param, value in grid_search.best_params_.items():
    if param != 'classifier':
        print(f"  - {param}: {value}")

# se guarda el ganador
output_model_path = 'data/best_linear_pipeline_grid.pkl'
joblib.dump(grid_search.best_estimator_, output_model_path)
print(f"\nEstimador campeón exportado a: {output_model_path}")


--- RESULTADOS DEL GRIDSEARCH K=5 ---
Mejor F1-Score Macro obtenido: 0.9031

Modelo Ganador: LinearSVC
Hiperparámetros óptimos:
  - classifier__C: 1.0
  - vectorizer__tfidf__max_features: 5000

Estimador campeón exportado a: data/best_linear_pipeline_grid.pkl
